In [ ]:
import os
import sqlite3
from PIL import Image, ImageStat
import cv2
import numpy as np
import pandas as pd

In [ ]:
# папка с картинками
IMGS_DIR = "imgs"

# БД
DB_PATH = "mydb.db"

print("Files in imgs:", os.listdir(IMGS_DIR)[:10])

Files in imgs: ['molotok.webp', 'molotok2.webp', 'molotok3.webp', 'molotok4.webp']


In [ ]:
def is_color(img):
    if img.mode != "RGB":
        return "black_white"
    stat = ImageStat.Stat(img.convert("RGB"))
    r, g, b = stat.mean
    if abs(r - g) < 10 and abs(g - b) < 10:
        return "neutral"
    return "colorful"

In [ ]:
def has_face(img_path):
    try:
        face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
        return len(faces) > 0
    except Exception as e:
        print(f"Face detection failed: {e}")
        return False

In [ ]:
def basic_features(img_path):
    img = Image.open(img_path)
    width, height = img.size
    file_size = os.path.getsize(img_path)

    return {
        "filename": os.path.basename(img_path),
        "width": width,
        "height": height,
        "file_size_kb": file_size / 1024,
        "color_mode": is_color(img),
        "presence_face": has_face(img_path),
        "text_on_image": False,
    }

In [ ]:
conn = sqlite3.connect(DB_PATH)

with conn:
    conn.execute('''
        CREATE TABLE IF NOT EXISTS wb_image_analysis (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            filename TEXT NOT NULL,
            width INTEGER,
            height INTEGER,
            file_size_kb REAL,
            color_mode TEXT,
            presence_face BOOLEAN,
            text_on_image BOOLEAN,
            sales_per_view REAL DEFAULT NULL
        );
    ''')

conn.close()

In [ ]:
analysis = []

for fname in os.listdir(IMGS_DIR):
    if not fname.lower().endswith((".jpg", ".png", ".jpeg", ".webp")):
        continue

    img_path = os.path.join(IMGS_DIR, fname)
    print(f"Analyzing {fname}...")
    features = basic_features(img_path)
    analysis.append(features)

df = pd.DataFrame(analysis)
print("\n📊 Пример фич:")
display(df.head())

# Подготовка: логические поля BOOLEAN → INTEGER (0/1)
df["presence_face"] = df["presence_face"].astype(int)
df["text_on_image"] = df["text_on_image"].astype(int)

# Запись в БД
conn = sqlite3.connect(DB_PATH)
df.to_sql("wb_image_analysis", conn, if_exists="replace", index=False)
conn.close()

print("✅ Results saved to wb_image_analysis in mydb.db")

Analyzing molotok.webp...
Analyzing molotok2.webp...
Analyzing molotok3.webp...
Analyzing molotok4.webp...

📊 Пример фич:


,filename,width,height,file_size_kb,color_mode,presence_face,text_on_image
0,molotok.webp,900,1200,128.822266,colorful,False,False
1,molotok2.webp,1800,2400,203.041016,colorful,True,False
2,molotok3.webp,900,1200,56.273438,colorful,False,False
3,molotok4.webp,900,1200,52.273438,colorful,True,False


✅ Results saved to wb_image_analysis in mydb.db
